In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/home/t/terencew/project-cluo/igvf/2023_YR2/multiome/'
donors = list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str))

In [ ]:
adata = sc.read_h5ad(f'{projdir}/h5ad/rna/processed/intersect_atac_reindex_small.h5ad')

In [ ]:
adata

In [ ]:
clusters = pd.read_csv(f'{projdir}/csv/clusters/rna_leiden_v7.csv',
                       sep='\t', header=0, index_col=0)
clusters.shape

In [ ]:
adata.obs['cluster'] = clusters.reindex(adata.obs.index)['clusters_v7']
adata.obs['coarse'] = clusters.reindex(adata.obs.index)['coarse_v7']

In [ ]:
sc.pl.umap(adata, color=['time_num', 'traj_score'])

In [ ]:
raw_adata = sc.read_h5ad(f'{projdir}/h5ad/rna/raw/intersect_atac.h5ad')

In [ ]:
raw_adata

In [ ]:
raw_adata.X[:100,:100].todense()

In [ ]:
raw_adata.obs['cluster'] = clusters.reindex(raw_adata.obs.index)['clusters_v7']
raw_adata.obs['coarse'] = clusters.reindex(raw_adata.obs.index)['coarse_v7']

In [ ]:
np.sum(raw_adata.obs.index == adata.obs.index)

In [ ]:
raw_adata.obs['coarse']

In [ ]:
whitelist = ['Start', 'Fail1']
mask = [x in whitelist for x in raw_adata.obs['coarse']]
tmp_adata = raw_adata[mask]

sc.pp.normalize_total(tmp_adata)
sc.pp.log1p(tmp_adata)

In [ ]:
tmp_adata

In [ ]:
sc.tl.rank_genes_groups(tmp_adata, 'coarse', method='wilcoxon', key_added = "wilcoxon")
sc.tl.rank_genes_groups(tmp_adata, 'coarse', method='t-test', key_added = "t-test")

In [ ]:
sc.pl.rank_genes_groups(tmp_adata, key='wilcoxon')

In [ ]:
sc.pl.rank_genes_groups(tmp_adata, key='t-test')

In [ ]:
start = sc.get.rank_genes_groups_df(tmp_adata, group='Start', key='wilcoxon')
fail1 = sc.get.rank_genes_groups_df(tmp_adata, group='Fail1', key='wilcoxon')

In [ ]:
start.head()

In [ ]:
fail1.head()

In [ ]:
start["tmp_pvals_adj"] = start["pvals_adj"].replace(0, 1e-300)
start["neglog10_pvals"] = -np.log10(start["tmp_pvals_adj"])

fail1["tmp_pvals_adj"] = fail1["pvals_adj"].replace(0, 1e-300)
fail1["neglog10_pvals"] = -np.log10(fail1["tmp_pvals_adj"])

In [ ]:
plt.figure(figsize=(6,6))

fc_thresh = 2
p_thresh = 1e-20

start["significant"] = (
    (abs(start["logfoldchanges"]) > fc_thresh) &
    (start["tmp_pvals_adj"] < p_thresh)
)

plt.scatter(
    start.loc[~start["significant"], "logfoldchanges"],
    start.loc[~start["significant"], "neglog10_pvals"],
    s=5,
    alpha=0.3
)

plt.scatter(
    start.loc[start["significant"], "logfoldchanges"],
    start.loc[start["significant"], "neglog10_pvals"],
    s=8
)

plt.axvline(fc_thresh, linestyle="--")
plt.axvline(-fc_thresh, linestyle="--")
plt.axhline(-np.log10(p_thresh), linestyle="--")

plt.xlabel("Log2 Fold Change")
plt.ylabel("-log10 adjusted p-value")
plt.axvline(0, color="black", linestyle="--")
plt.show()


In [ ]:
plt.figure(figsize=(6,6))

fc_thresh = 2
p_thresh = 1e-20

fail1["significant"] = (
    (abs(fail1["logfoldchanges"]) > fc_thresh) &
    (fail1["tmp_pvals_adj"] < p_thresh)
)

plt.scatter(
    fail1.loc[~fail1["significant"], "logfoldchanges"],
    fail1.loc[~fail1["significant"], "neglog10_pvals"],
    s=5,
    alpha=0.3
)

plt.scatter(
    fail1.loc[fail1["significant"], "logfoldchanges"],
    fail1.loc[fail1["significant"], "neglog10_pvals"],
    s=8
)

plt.axvline(fc_thresh, linestyle="--")
plt.axvline(-fc_thresh, linestyle="--")
plt.axhline(-np.log10(p_thresh), linestyle="--")

plt.xlabel("Log2 Fold Change")
plt.ylabel("-log10 adjusted p-value")
plt.axvline(0, color="black", linestyle="--")
plt.show()


In [ ]:
mask = fail1['pvals_adj'] < 1e-20
mask.sum()

### volcano plot

In [ ]:
start.to_csv(f'{projdir}/csv/degs/start_vs_fail1.csv', sep='\t')
fail1.to_csv(f'{projdir}/csv/degs/fail1_vs_start.csv', sep='\t')

### visualize

In [ ]:
raw_adata = sc.read_h5ad(f'{projdir}/h5ad/rna/raw/intersect_atac.h5ad')
raw_adata.obs = adata.obs.copy()
raw_adata.obsm = adata.obsm.copy()
sc.pp.normalize_total(raw_adata)
sc.pp.log1p(raw_adata)

In [ ]:
sc.pl.umap(raw_adata, color='coarse')

In [ ]:
n = 20
genes = start['names'][:n]
genes = genes[genes.isin(adata.var.index)]

sc.pl.umap(raw_adata, color=genes, ncols=5, vmax=2)

In [ ]:
n = 20
genes = fail1['names'][:n]
genes = genes[genes.isin(adata.var.index)]

sc.pl.umap(raw_adata, color=genes, ncols=5)

In [ ]:
!date